### Models for Milestone 3

In [16]:
#Librariesfrom sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, recall_score, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import pandas as pd

In [12]:
parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

#Drop nas #
df = pd.read_parquet(parquet_path)
df = df.dropna()
df['target'] = (df['conflict_count'] >= 1).astype(int)
features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
features = pd.get_dummies(features, columns=['year'], prefix='year')
X = features
y = df['target']

In [13]:

# Preprocess data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train with gradient descent
# 'saga' is a variant of stochastic gradient descent
model = LogisticRegression(
    solver='saga',  # SGD variant
    penalty='l2',   # L2 regularization
    C=1.0,          # Regularization strength (inverse)
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9299


### Concerned about class imbalance

In [17]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
model_smote = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=1.0,
    max_iter=1000,
    random_state=42
)
model_smote.fit(X_train_smote, y_train_smote)

print("\n--- SMOTE Model ---")
y_pred_smote = model_smote.predict(X_test_scaled)
print(f"Accuracy: {model_smote.score(X_test_scaled, y_test):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_smote):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))


--- SMOTE Model ---
Accuracy: 0.8521
Recall: 0.7536

Confusion Matrix:
[[5381  870]
 [ 136  416]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.86      0.91      6251
           1       0.32      0.75      0.45       552

    accuracy                           0.85      6803
   macro avg       0.65      0.81      0.68      6803
weighted avg       0.92      0.85      0.88      6803

